In [1]:
import os

DATA_DIR = "/kaggle/input"

for name in os.listdir(DATA_DIR):
    print(name)

competitions


In [2]:
import os

DATA_DIR = "/kaggle/input/competitions"

for name in os.listdir(DATA_DIR):
    print(name)

stanford-rna-3d-folding


In [3]:
# Replace this with the folder name you saw in Step 1
COMPETITION_DIR = "/kaggle/input/competitions/stanford-rna-3d-folding"

# Check the important files and folders
expected = [
    "train_sequences.csv",
    "train_labels.csv",
    "validation_sequences.csv",
    "validation_labels.csv",
    "PDB_RNA",
]

for name in expected:
    path = os.path.join(COMPETITION_DIR, name)
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'NOT FOUND'}")

train_sequences.csv: FOUND
train_labels.csv: FOUND
validation_sequences.csv: FOUND
validation_labels.csv: FOUND
PDB_RNA: FOUND


In [4]:
import pandas as pd

seq_path = os.path.join(COMPETITION_DIR, "train_sequences.csv")
labels_path = os.path.join(COMPETITION_DIR, "train_labels.csv")

sequences = pd.read_csv(seq_path)
labels = pd.read_csv(labels_path)

print("Sequences shape:", sequences.shape)
print("Sequence columns:", sequences.columns.tolist())
display(sequences.head())

print("Labels shape:", labels.shape)
print("Label columns:", labels.columns.tolist())
display(labels.head())

Sequences shape: (844, 5)
Sequence columns: ['target_id', 'sequence', 'temporal_cutoff', 'description', 'all_sequences']


,target_id,sequence,temporal_cutoff,description,all_sequences
0,1SCL_A,GGGUGCUCAGUACGAGAGGAACCGCACCC,1995-01-26,"THE SARCIN-RICIN LOOP, A MODULAR RNA",>1SCL_1|Chain A|RNA SARCIN-RICIN LOOP|Rattus n...
1,1RNK_A,GGCGCAGUGGGCUAGCGCCACUCAAAAGGCCCAU,1995-02-27,THE STRUCTURE OF AN RNA PSEUDOKNOT THAT CAUSES...,>1RNK_1|Chain A|RNA PSEUDOKNOT|null\nGGCGCAGUG...
2,1RHT_A,GGGACUGACGAUCACGCAGUCUAU,1995-06-03,24-MER RNA HAIRPIN COAT PROTEIN BINDING SITE F...,>1RHT_1|Chain A|RNA (5'-R(P*GP*GP*GP*AP*CP*UP*...
3,1HLX_A,GGGAUAACUUCGGUUGUCCC,1995-09-15,P1 HELIX NUCLEIC ACIDS (DNA/RNA) RIBONUCLEIC ACID,>1HLX_1|Chain A|RNA (5'-R(*GP*GP*GP*AP*UP*AP*A...
4,1HMH_E,GGCGACCCUGAUGAGGCCGAAAGGCCGAAACCGU,1995-12-07,THREE-DIMENSIONAL STRUCTURE OF A HAMMERHEAD RI...,">1HMH_1|Chains A, C, E|HAMMERHEAD RIBOZYME-RNA..."


Labels shape: (137095, 6)
Label columns: ['ID', 'resname', 'resid', 'x_1', 'y_1', 'z_1']


,ID,resname,resid,x_1,y_1,z_1
0,1SCL_A_1,G,1,13.760,-25.974001,0.102
1,1SCL_A_2,G,2,9.310,-29.638000,2.669
2,1SCL_A_3,G,3,5.529,-27.813000,5.878
3,1SCL_A_4,U,4,2.678,-24.900999,9.793
4,1SCL_A_5,G,5,1.827,-20.136000,11.793


In [5]:
!pip install -q gemmi


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 20.4 MB/s eta 0:00:0000:0100:01


In [6]:
!pip install -q gemmi riboseek
!riboseek --help
!riboseek encode --help

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 973.5 kB/s eta 0:00:00 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 18.3 MB/s eta 0:00:0000:0100:01
usage: riboseek [-h] [--version] {encode,search,build-db,download-db,info} ...

Ultra-fast RNA 3D structure search via spatial-neighbor structural alphabets.

positional arguments:
  {encode,search,build-db,download-db,info}
    encode              encode a PDB / mmCIF into RS-20 letters
    search              search the database
    build-db            encode a directory of structures
    download-db         fetch the full 15,391-chain database
    info                show version / status

options:
  -h, --help            show this help message and exit
  --version             show program's version number and exit
usage: riboseek encode [-h] [--chain CHAIN] [--alphabet ALPHABE

In [7]:
import pandas as pd, numpy as np, gemmi, json, subprocess, re
from pathlib import Path
from scipy.spatial import cKDTree
from multiprocessing import Pool

DATA = next(Path("/kaggle/input").rglob("train_sequences.v2.csv")).parent
SPL = (list(Path("/kaggle/working").glob("splits.tsv")) + list(Path("/kaggle/input").rglob("splits.tsv")))[0]
splits = pd.read_csv(SPL, sep="\t")
chains = pd.read_csv(SPL.parent/"chain_list.tsv", sep="\t")
df = chains.merge(splits[["chain_id","family","split","borderline"]], on="chain_id")
df = df[df.split != "excluded"].reset_index(drop=True)
print(df.split.value_counts().to_dict(), "| has_cif:", df.has_cif.sum())

OUT = Path("/kaggle/working/alizeh"); CH_DIR = OUT/"chains"
CH_DIR.mkdir(parents=True, exist_ok=True)
BASES = "ACGU"

{'train': 1606, 'test': 184, 'val': 160} | has_cif: 1950


In [8]:
lab = pd.read_csv(DATA/"train_labels.v2.csv", usecols=["ID","resname","resid","x_1","y_1","z_1"])
lab["chain_id"] = lab.ID.str.rsplit("_", n=1).str[0]
lab = lab[lab.chain_id.isin(df.chain_id)]
labels = {k: g.sort_values("resid").reset_index(drop=True) for k, g in lab.groupby("chain_id")}
del lab

bad = [cid for cid, s in zip(df.chain_id, df.sequence)
       if len(labels[cid]) != len(s) or "".join(labels[cid].resname) != s
       or not (labels[cid].resid.to_numpy() == np.arange(1, len(s)+1)).all()]
print("label/sequence inconsistencies:", len(bad), bad[:5])

label/sequence inconsistencies: 0 []


In [14]:
TOL = 0.05   # Å; Kaggle C1' coords should equal the cif coords

def parent_base(resname):
    """Residue name in the cif -> base letter (A/C/G/U). Modified bases map to their parent."""
    info = gemmi.find_tabulated_residue(resname)
    if info is None or not info.is_nucleic_acid(): return "?"
    b = info.one_letter_code.upper()
    return b if b in BASES else "?"

def load_model(pdb_id):
    """Open a .cif and clean it. Keeps ALL models (NMR files have many)."""
    st = gemmi.read_structure(str(DATA/"PDB_RNA"/f"{pdb_id}.cif"))
    st.setup_entities()
    st.remove_alternative_conformations(); st.remove_hydrogens()
    st.remove_ligands_and_waters(); st.remove_empty_chains()
    return st

def _c1_index(model):
    """List every C1' atom in one model, with its (chain index, residue index)."""
    recs, coords = [], []
    for ci, ch in enumerate(model):
        for k, r in enumerate(ch):
            a = r.find_atom("C1'", "*")
            if a is not None:
                recs.append((ci, k)); coords.append((a.pos.x, a.pos.y, a.pos.z))
    return np.array(recs), np.array(coords)

def map_chain(st, cid, seq):
    """Find the model + chain + residue for each sequence position by matching C1' coordinates."""
    xyz = labels[cid][["x_1","y_1","z_1"]].to_numpy()
    resolved = ~np.isnan(xyz[:, 0])
    if not resolved.any(): return None, "no_resolved_positions"
    best = None
    for mi, model in enumerate(st):                       # try every model
        recs, coords = _c1_index(model)
        if len(coords) == 0: continue
        d, idx = cKDTree(coords).query(xyz[resolved])
        rate = (d < TOL).mean()
        if best is None or rate > best[0]: best = (rate, mi, recs, d, idx)
        if rate > 0.99: break                             # found it, stop searching
    if best is None: return None, "no_C1'_atoms"
    rate, mi, recs, d, idx = best
    if rate < 0.95: return None, f"coord_match={rate:.2f} median_dist={np.median(d):.2f}A"
    hit = d < TOL
    ci = np.bincount(recs[idx[hit], 0]).argmax()
    if (recs[idx[hit], 0] == ci).mean() < 0.99: return None, "positions_span_chains"
    k_of_pos = np.full(len(seq), -1)
    good = hit & (recs[idx, 0] == ci)
    k_of_pos[np.where(resolved)[0][good]] = recs[idx[good], 1]
    ks = k_of_pos[k_of_pos >= 0]
    if len(set(ks)) != len(ks): return None, "many_to_one"
    if not (np.diff(ks) > 0).all(): return None, "order_mismatch"
    ch = st[mi][int(ci)]
    mism = sum(parent_base(ch[int(k)].name) != seq[p] for p, k in enumerate(k_of_pos) if k >= 0)
    if mism > 0.02 * len(ks): return None, f"base_mismatch={mism}"
    return (int(mi), int(ci), k_of_pos), f"ok (model {mi+1} of {len(st)})"

def write_chain(st, mi, ci, path):
    """Save only the matched model + chain as its own small .pdb file."""
    s = st.clone()
    if mi + 1 < len(s): del s[mi+1:]
    if mi > 0: del s[:mi]
    for i in reversed(range(len(s[0]))):
        if i != ci: del s[0][i]
    s[0][0].name = "A"
    s.write_pdb(str(path))
    return list(s[0][0])

def riboseek_rs20(path):
    """Run RiboSeek on one chain file. Returns (list of RS-20 numbers 0-19, sequence RiboSeek read)."""
    out = subprocess.run(["riboseek", "encode", "--alphabet", "sa20", "--format", "labels",
                          "--keep-modified", str(path)], capture_output=True, text=True, timeout=300)
    if out.returncode != 0: raise RuntimeError(out.stderr[-300:])
    labs = [int(x) for x in out.stdout.strip().split(",") if x.strip()]
    m = re.search(r"sequence=(\S*)", out.stderr)
    return labs, (m.group(1) if m else "")

In [13]:
# A) do the two NMR chains map now?
for cid in ["1ULL_A", "2K96_A"]:
    r = df[df.chain_id == cid].iloc[0]
    st = load_model(r.pdb_id)
    print(cid, "| models in file:", len(st), "| mapping:", map_chain(st, cid, r.sequence)[1])

# B) what do RiboSeek's output options look like?
r = df[df.chain_id == "8DC2_B"].iloc[0]
st = load_model(r.pdb_id)
(mi, ci, k_of_pos), info = map_chain(st, r.chain_id, r.sequence)
residues = write_chain(st, mi, ci, CH_DIR/"probe.pdb")
print("\nprobe chain residues:", len(residues), [(x.name, x.seqid.num) for x in residues[:5]])
!riboseek info
for args in [["--format", "labels"], ["--keep-modified"],
             ["--alphabet", "sa20", "--format", "labels"],
             ["--alphabet", "rs80"], ["--alphabet", "rs80", "--format", "labels"]]:
    o = subprocess.run(["riboseek", "encode", *args, str(CH_DIR/"probe.pdb")],
                       capture_output=True, text=True)
    print("\n###", args, "| return code", o.returncode)
    print("STDOUT:", o.stdout[:600])
    if o.stderr.strip(): print("STDERR:", o.stderr[-300:])

1ULL_A | models in file: 7 | mapping: ok (model 7 of 7)
2K96_A | models in file: 20 | mapping: ok (model 20 of 20)

probe chain residues: 51 [('U', -34), ('U', -33), ('G', -32), ('U', -31), ('U', -30)]
riboseek 0.2.0
  bundled alphabets : rs80 (default), sa20 (= RS-20)
  cached full db    : /root/.cache/riboseek/encoded_chains.json  (not downloaded)
  C extension       : ok

### ['--format', 'labels'] | return code 0
STDOUT: 8,5,1,15,4,13,13,9,13,13,13,13,13,15,13,19,5,5,18,10,5,15,6,19,1,4,13,13,4,12,15,15,13,13,5,5,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1

STDERR: # chain=A length=51 sequence=UUGUUGUAACUCUUAUUUUGUAUGGAGUAAACAACUAGCAUCACCUUCACC


### ['--keep-modified'] | return code 0
STDOUT: IFBPENNJNNNNNPNTFFSKFPGTBENNEMPPNNFFBBBBBBBBBBBBBBB

STDERR: # chain=A length=51 sequence=UUGUUGUAACUCUUAUUUUGUAUGGAGUAAACAACUAGCAUCACCUUCACC


### ['--alphabet', 'sa20', '--format', 'labels'] | return code 0
STDOUT: 8,5,1,15,4,13,13,9,13,13,13,13,13,15,13,19,5,5,18,10,5,15,6,19,1,4,13,13,4,12,15,15,13,13,

In [15]:
import riboseek, pathlib
pkg = pathlib.Path(riboseek.__file__).parent
for f in sorted(pkg.rglob("*.py")):
    for i, line in enumerate(f.read_text().splitlines(), 1):
        if re.search(r"\*\s*4\b|ACGU|base_to|BASE_", line):
            print(f"{f.name}:{i}: {line.strip()}")

alphabet.py:103: BASE_INDEX = {"A": 0, "C": 1, "G": 2, "U": 3, "T": 3}
alphabet.py:104: BASE_ALPHA = 2.0
alphabet.py:108: """80-letter composite alphabet: ``J = RS20 * 4 + base``.
alphabet.py:116: def __init__(self, base: Alphabet, alpha: float = BASE_ALPHA):
alphabet.py:145: base[i] = BASE_INDEX.get(seq[i], 0)
alphabet.py:146: return (lab * 4 + base).astype(np.int32)
features.py:40: _BASE_ATOMS = {
features.py:141: if resname and resname in _BASE_ATOMS:
features.py:142: pts = [atom_xyz[a] for a in _BASE_ATOMS[resname] if a in atom_xyz]


In [16]:
def process_entry(args):
    pdb_id, rows = args
    out = []
    try: st = load_model(pdb_id)
    except Exception as e:
        return [dict(chain_id=c, split=sp, status="cif_read_error", detail=str(e)[:200])
                for c, _, sp, *_ in rows]
    for cid, seq, split, fam, bl, rel in rows:
        rec = dict(chain_id=cid, pdb_id=pdb_id, split=split, family=fam, borderline=bl,
                   release_date=rel, sequence=seq, status="ok", detail="")
        # 1) match sequence positions to residues
        m, info = map_chain(st, cid, seq)
        if m is None: rec.update(status="map_fail", detail=info); out.append(rec); continue
        mi, ci, k_of_pos = m
        # 2) save the chain as its own file
        path = CH_DIR/f"{cid}.pdb"
        residues = write_chain(st, mi, ci, path)
        # 3) run RiboSeek
        try: labs, tool_seq = riboseek_rs20(path)
        except Exception as e: rec.update(status="encode_error", detail=str(e)[:200]); out.append(rec); continue
        if len(labs) != len(residues):
            rec.update(status="encode_len_mismatch", detail=f"{len(labs)} tokens vs {len(residues)} residues")
            out.append(rec); continue
        # 4) check RiboSeek read the same bases we did
        our_bases = [parent_base(x.name) for x in residues]
        if len(tool_seq) == len(residues):
            n_diff = sum(a != b for a, b in zip(our_bases, tool_seq) if a in BASES)
            if n_diff:
                rec.update(status="tool_sequence_mismatch", detail=f"{n_diff} bases differ"); out.append(rec); continue
            chain_bases = list(tool_seq)
        else:
            chain_bases = our_bases
        # 5) per-position labels, same length as the sequence (-1 = no token)
        tok_k = [t if 0 <= t < 20 else -1 for t in labs]
        rs20 = [tok_k[k] if k >= 0 else -1 for k in k_of_pos]
        rs80 = [t*4 + BASES.index(b) if t >= 0 else -1 for t, b in zip(rs20, seq)]
        rec.update(rs20=rs20, rs80=rs80, coverage=float(np.mean(np.array(rs20) >= 0)),
                   cif_path=str(path), auth_chain=st[mi][ci].name, model_used=mi + 1,
                   rs20_chain="".join(chr(65 + t) if t >= 0 else "X" for t in tok_k),
                   rs80_chain=" ".join(str(t*4 + BASES.index(b)) if t >= 0 and b in BASES else "-1"
                                       for t, b in zip(tok_k, chain_bases)),
                   map_rows=[(p + 1, seq[p], int(k),
                              residues[k].seqid.num if k >= 0 else None,
                              residues[k].seqid.icode.strip() if k >= 0 else "",
                              residues[k].name if k >= 0 else "",
                              bool(k >= 0 and residues[k].name not in BASES),
                              rs20[p], rs80[p])
                             for p, k in enumerate(k_of_pos)])
        if rec["coverage"] < 0.7: rec["status"] = "low_token_coverage"
        out.append(rec)
    return out

todo = df[df.has_cif]
jobs = [(pid, list(g[["chain_id","sequence","split","family","borderline","release_date"]]
                   .itertuples(index=False, name=None))) for pid, g in todo.groupby("pdb_id")]
results = []
with Pool(4) as pool:
    for i, r in enumerate(pool.imap_unordered(process_entry, jobs, chunksize=4)):
        results += r
        if i % 200 == 0: print(i, "/", len(jobs), "entries done")
res = pd.DataFrame(results)
print(pd.crosstab(res.status, res.split))
print(res[res.status != "ok"].detail.str.split("=").str[0].value_counts().head(10))

0 / 1742 entries done
200 / 1742 entries done
400 / 1742 entries done
600 / 1742 entries done
800 / 1742 entries done
1000 / 1742 entries done
1200 / 1742 entries done
1400 / 1742 entries done
1600 / 1742 entries done
split                   test  train  val
status                                  
encode_len_mismatch        8     25    1
map_fail                   0     25    0
ok                       176   1549  159
tool_sequence_mismatch     0      7    0
detail
base_mismatch               24
3 bases differ               7
76 tokens vs 77 residues     5
72 tokens vs 73 residues     3
23 tokens vs 25 residues     3
73 tokens vs 74 residues     2
75 tokens vs 76 residues     2
19 tokens vs 20 residues     1
positions_span_chains        1
34 tokens vs 38 residues     1
Name: count, dtype: int64


In [17]:
ok = res[res.status == "ok"].copy()
assert ok.chain_id.is_unique
assert (ok.sequence.str.len() == ok.rs20.str.len()).all()
print("usable chains:", ok.split.value_counts().to_dict())
print("coverage:", ok.coverage.describe()[["mean","25%","50%"]].round(3).to_dict())
print("modified-base positions:", sum(sum(m[6] for m in rows) for rows in ok.map_rows))

def flat(sp, col):
    X, Y = [], []
    for s, t in zip(ok[ok.split == sp].sequence, ok[ok.split == sp][col]):
        for b, y in zip(s, t):
            if y >= 0: X.append(b); Y.append(y)
    return np.array(X), np.array(Y)

for col, K in [("rs20", 20), ("rs80", 80)]:
    Xtr, Ytr = flat("train", col); Xva, Yva = flat("val", col)
    maj = np.bincount(Ytr, minlength=K).argmax()
    nuc = {b: np.bincount(Ytr[Xtr == b], minlength=K).argmax() for b in BASES}
    print(f"{col}: classes used {len(np.unique(Ytr))}/{K} | val majority acc {(Yva == maj).mean():.3f}"
          f" | val nucleotide-only acc {np.mean([nuc[b] == y for b, y in zip(Xva, Yva)]):.3f}")
print("RS-20 train distribution:", np.round(np.bincount(flat('train','rs20')[1], minlength=20)
                                            / len(flat('train','rs20')[1]), 3))

usable chains: {'train': 1549, 'test': 176, 'val': 159}
coverage: {'mean': 0.976, '25%': 0.975, '50%': 1.0}
modified-base positions: 0
rs20: classes used 20/20 | val majority acc 0.183 | val nucleotide-only acc 0.202
rs80: classes used 80/80 | val majority acc 0.054 | val nucleotide-only acc 0.202
RS-20 train distribution: [0.001 0.122 0.038 0.003 0.146 0.072 0.027 0.095 0.023 0.047 0.026 0.012
 0.019 0.177 0.025 0.078 0.019 0.012 0.017 0.038]


In [18]:
# Haleema: sequence -> per-position labels (-1 = no reliable token; use ignore_index=-1)
with open(OUT/"dataset.jsonl", "w") as f:
    for r in ok.itertuples():
        f.write(json.dumps(dict(chain_id=r.chain_id, split=r.split, family=int(r.family),
                                borderline=bool(r.borderline), sequence=r.sequence,
                                rs20=r.rs20, rs80=r.rs80)) + "\n")

# Ansa: template DB + residue mapping
ok[["chain_id","pdb_id","auth_chain","split","family","release_date","cif_path",
    "rs20_chain","rs80_chain","coverage"]].assign(length=ok.sequence.str.len()) \
  .to_csv(OUT/"template_db.tsv", sep="\t", index=False)
pd.DataFrame([(r.chain_id, *m) for r in ok.itertuples() for m in r.map_rows],
             columns=["chain_id","seq_pos","base","cif_res_index","auth_seq_id","icode",
                      "cif_resname","is_modified","rs20","rs80"]) \
  .to_csv(OUT/"residue_map.tsv", sep="\t", index=False)

# everyone: why each chain was dropped
res[["chain_id","split","status","detail"]].to_csv(OUT/"qc_log.tsv", sep="\t", index=False)
print(sorted(p.name for p in OUT.iterdir()), "| chain files:", len(list(CH_DIR.iterdir())))

['chains', 'dataset.jsonl', 'qc_log.tsv', 'residue_map.tsv', 'template_db.tsv'] | chain files: 1926


In [19]:
!cd /kaggle/working && zip -rq alizeh.zip alizeh && ls -lh alizeh.zip

-rw-r--r-- 1 root root 90M Oct 21 04:27 alizeh.zip


In [20]:
import os
print(os.getcwd())
print(os.path.abspath("alizeh.zip"))

/kaggle/working
/kaggle/working/alizeh.zip
